# ADEPT Architectural Analysis Journey: Anti_Corruption_LayerThis notebook provides a comprehensive walkthrough of the ADEPT framework's analysis pipeline.

In [ ]:
%matplotlib inline
import os
import json
import itertools
import pandas as pd
import matplotlib.pyplot as plt
from adept import PatternAnalysis

# Set project root in path so we can import 'adept' package
import sys
sys.path.append("../../")

## 1. Initialize and Load Data

In [ ]:
json_path = "Anti_Corruption_Layer.json"
session = PatternAnalysis(json_path)

# Load both definition and data
session.load(validate_integrity=True)

print(f"Loaded {len(session.raw_df)} experiments.")
session.raw_df.head()

## 2. Define Tradeoffs (Programmatic Fallback)

In [ ]:
if not session.get_tradeoffs():
    print("No tradeoffs defined in JSON. Defining programmatically...")
    session.add_tradeoff(
        name="performance-optimized",
        elements={"response_time": "fast", "utilization": "average"}
    )
    session.add_tradeoff(
        name="balanced",
        elements={"response_time": "average", "utilization": "average"}
    )

# Discretize outcomes
target_labels = {qa.name: ['fast', 'average', 'slow'] if 'time' in qa.name else ['low', 'average', 'high'] 
                 for qa in session.get_outcomes()}
discrete_df, schemes = session.define_tradeoffs(n_bins=3, labels=target_labels, method='discretization')

for scheme in schemes:
    print(f"Objective: {scheme.objective_name}")
    for b in scheme.bins:
        print(f"  {b.label}: [{b.min_value:.2f}, {b.max_value:.2f}]")

## 3. Data Splitting

In [ ]:
session.split_data(test_size=0.2)
print(f"Train size: {len(session.train_indices)}")
print(f"Test size: {len(session.test_indices)}")

## 4. Outcome Distributions

In [ ]:
for tradeoff in session.get_tradeoffs():
    indices = session.get_indices_for_tradeoff(tradeoff)
    print(f"Plotting distribution for: {tradeoff.name}")
    session.coordinator.plot_distributions(
        session.outcomes_df, 
        schemes, 
        tradeoff=tradeoff,
        highlight_indices=indices
    )
    plt.show()

## 5. Quality Objective Space (2D Scatter)

In [ ]:
outcome_cols = list(session.outcomes_df.columns)
for x_col, y_col in itertools.combinations(outcome_cols, 2):
    session.show_quality_objective_space(
        x_col, y_col, 
        highlight_tradeoffs=session.get_tradeoffs(),
        show_overall=True,
        subset='all'
    )
    plt.show()

## 6. Contingency Analysis

In [ ]:
decisions = list(session.get_decisions().keys())
for decision_key in decisions:
    print(f"Analyzing Decision: {decision_key}")
    session.show_policy_contingency(decision_key, type='heatmap', subset='train')
    plt.show()

## 7. Robustness Analysis

In [ ]:
for tradeoff in session.get_tradeoffs():
    print(f"Robustness Ranking for Tradeoff: {tradeoff.name}")
    ranking = session.get_policy_robustness_ranking(tradeoff.name, metric='starr')
    for i, (pol, val) in enumerate(ranking[:3]):
        print(f"  {i+1}. {pol}: {val:.2f}% success")
    
    if ranking:
        top_pol = ranking[0][0]
        regret = session.compute_robustness(top_pol, tradeoff.name, metric='regret')
        print(f"  Top Policy '{top_pol}' Regret: {regret.get('value', 0.0):.4f}")

print("
Overall Robustness Heatmap (STARR):")
session.show_robustness_heatmap(metric='starr')
plt.show()

## 8. Feature Importance

In [ ]:
scores_df = session.compute_feature_scores(use_smart_correlation=True, subset='train')
session.show_feature_heatmap(scores_df, title="Feature Influence (Train Set)")
plt.show()

## 9. Scenario Discovery (PRIM)

In [ ]:
for tradeoff in session.get_tradeoffs():
    print(f"Discovering scenarios for: {tradeoff.name}")
    boxes = session.discover_scenarios(tradeoff.name, method='prim', threshold=0.7, standardize=True)
    if boxes:
        best_box = boxes[0]
        print(f"  Key Rules: {best_box.limits}")
        print(f"  Metrics: {best_box.metrics}")
    else:
        print("  No stable scenarios discovered.")

## 10. Global Discovery (CART)

In [ ]:
boxes_cart = session.discover_scenarios(method='cart', standardize=False)
for box in boxes_cart:
    if box.metrics.get('targets_in_box', 0) > 10:
        print(f"Box for {box.target_tradeoff}: {box.limits}")